In [ ]:
!pip install -r requirement.txt



ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirement.txt'


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import pandas as pd
import seaborn as sns

In [ ]:
df = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

In [ ]:
from sklearn.model_selection import train_test_split
X = df.iloc[:,:-1]
Y = df.iloc[:,-1]
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size = 0.1, random_state = 42)

In [ ]:
def scale_with_names(XTrain, XTest, scaler = None):
    if scaler is None:
        from sklearn.preprocessing import MinMaxScaler
        scaler = MinMaxScaler()
    arr1 = scaler.fit_transform(XTrain)
    XTrain = pd.DataFrame(arr1, columns = XTrain.columns, index = XTrain.index)
    arr2 = scaler.transform(XTest)
    XTest = pd.DataFrame(arr2, columns = XTest.columns, index = XTest.index)
    return XTrain, XTest



In [ ]:
# Now we can just perform Min Max Scaling on the TrackDuration Column

df_new = X_train[['TrackDurationMs']]
df_new_test = df_test[['TrackDurationMs']]

df_new, df_new_test = scale_with_names(df_new, df_new_test)

X_train['TrackDurationMs'] = df_new['TrackDurationMs']
df_test['TrackDurationMs'] = df_new_test['TrackDurationMs']





In [ ]:
# Now we need to decide upon the Model that we are supposed to use
"""
We can go for many Regression Models, Lasso, Ridge and even Linear.
We will go through them all evaluating the model on the basis of the Loss function defined in the contest
and evaluate the model finally using RandomForestRegressor as well
"""

# Let's first use Ridge Regression

from sklearn.metrics import mean_squared_error

from sklearn.metrics import mean_absolute_percentage_error


from sklearn.linear_model import Ridge
model = Ridge()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred)

print("RMSE : ", rmse)

mape = mean_absolute_percentage_error(y_test, y_pred)

print("MAPE: ", mape)





RMSE :  76498171167.6714
MAPE:  2387.830795404285


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import BaggingRegressor

base_tree = DecisionTreeRegressor(random_state = 42)
bagging = BaggingRegressor(estimator = base_tree,
                           random_state = 42,
                           n_jobs = -1
)





In [ ]:
param_grids ={
    'n_estimators': [50,20,5],
    'max_samples': [0.5,0.8,1.0],
    'estimator__max_depth':[None,],
    'estimator__min_samples_split': [2]
}


In [ ]:
grid_search = GridSearchCV(
    estimator = bagging,
    param_grid = param_grids,
    scoring = "neg_mean_squared_error",
    cv = 5,
    n_jobs = -1
)

In [ ]:
 #grid_search.fit(X_train, y_train)

In [ ]:
# y_pred = grid_search.predict(df_test)

In [ ]:
'''submission = pd.read_csv("sample_submission.csv")
submission["BeatsPerMinute"] = y_pred
'''

'submission = pd.read_csv("sample_submission.csv")\nsubmission["BeatsPerMinute"] = y_pred\n'

In [ ]:
# submission.to_csv("DT_Regressor.csv", index = False)

In [ ]:
# Now we need to apply XG Boost here

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

boostreg = GradientBoostingRegressor(loss = 'squared_error', learning_rate = 0.001, n_estimators = 1000, subsample = 0.9, min_samples_leaf = 15, verbose = 1)

In [ ]:
X_fit = boostreg.fit(X_train, y_train)


from sklearn.metrics import mean_squared_error
y_pred = X_fit.predict(X_test)

rmse = mean_squared_error(y_test, y_pred)

print("RMSE : ", rmse)

      Iter       Train Loss      OOB Improve   Remaining Time 
         1         701.2336           0.0008           74.85m
         2         701.2200          -0.1130           69.61m
         3         700.7864          -3.8940           64.72m
         4         700.6810          -0.9393           62.30m
         5         700.5237          -1.4060           56.79m
         6         700.5973           0.6714           52.91m
         7         700.3874          -1.8808           50.09m
         8         701.3278           8.4735           47.94m
         9         700.8233          -4.5320           46.43m
        10         701.2353           3.7176           46.11m
        20         700.6954          -3.6182           39.84m
        30         700.5821           0.9472           38.00m
        40         700.6832           0.5431           36.83m
        50         700.6167          -0.7235           36.12m
        60         700.6553           4.3069           35.36m
       

In [ ]:
y_pred = X_fit.predict(df_test)

In [ ]:
submission = pd.read_csv("sample_submission.csv")
submission["BeatsPerMinute"] = y_pred

In [ ]:
submission.to_csv("Boosting_Regressor3.csv", index = False)

In [ ]:
'''from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.linear_model import LinearRegression

model  = Ridge()
model2 = Lasso()
model3 = RandomForestRegressor()
model4 =
 BaggingRegressor(estimator = None, n_estimators = 100, max_samples = 0.5)
model5 = LinearRegression()

model5.fit(X_train, y_train)
y_pred = model5.predict(df_test)
'''


'from sklearn.linear_model import Ridge\nfrom sklearn.linear_model import Lasso\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.ensemble import BaggingRegressor\nfrom sklearn.linear_model import LinearRegression\n\nmodel  = Ridge()\nmodel2 = Lasso()\nmodel3 = RandomForestRegressor()\nmodel4 =\n BaggingRegressor(estimator = None, n_estimators = 100, max_samples = 0.5)\nmodel5 = LinearRegression()\n\nmodel5.fit(X_train, y_train)\ny_pred = model5.predict(df_test)\n'